In [2]:
from sympy import *
from sympy.physics.quantum import TensorProduct as TP
import itertools as it

In [13]:
#we first define symbols for the elements of S_3 and for the equality patterns
E, T1, T2, T3, C1, C2, Eq, D = symbols('E, T_1, T_2, T_3, C_1, C_2, Eq, D')
#and then the set of permutations S_3, that is, the group
G = (E, T1, T2, T3, C1, C2)
Pi= (D, T1, T2, T3, Eq)
# Let us define the representation gamma (here called gam) - how the group S_3 acts on 
# 5-dim module R[\Pi] = span{ distinct | 12 eq, 23 eq, 31 eq | all equal}
e5 = eye(5)
gam = { E: e5,
        T1: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        T2: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        T3: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        C1: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
       #
        C2: Matrix([e5[0,:],  #[1,0,0,0,0]
                    e5[2,:],  #[0,0,1,0,0]
                    e5[3,:],  #[0,0,0,1,0]
                    e5[1,:],  #[0,1,0,0,0]
                    e5[4,:]]),#[0,0,0,0,1]
}

In [4]:
# The projector onto invariant subspace U = R[\Pi]^{S_3} w. r. to the decomposition R[\Pi]= U \oplus W 
#is constructed from the formula P_U = \frac 1{\#G} \sum_{g \in G} gam(g)
PU = Rational(1,6) * sum(gam.values(),zeros(5,5))
# A projector onto subspace W - orthogonal complement of U
PW = eye(5) - PU

e,t,c,s = symbols('e,t,c,s') # additional symbol: s = e+3*t+2*c
#for each party we write down the matrix mapping from the space R[S_3] to R[\Pi]
S = Matrix([[1,0,0,0,0,0],
            [1,1,0,0,0,0],
            [1,0,1,0,0,0],
            [1,0,0,1,0,0],
            [1,1,1,1,1,1]])
#and the corresponding Weingarten matrix
Weing = Matrix([[e,t,t,t,c,c],
            [t,e,c,c,t,t],
            [t,c,e,c,t,t],
            [t,c,c,e,t,t],
            [c,t,t,t,e,c],
            [c,t,t,t,c,e]])

#for a single subsystem, the coefficient matrix is defined as
SW = S*Weing[:,:-1] # last column removed
SW = SW.subs(e+3*t+2*c,s)

In [12]:
# We proceed to an explicit construction of the n-partite problem by setting the symbolic 
# vectors of invariants and equality patterns
# here and in the following lines, n will be used to denote the number of subsystems
def x_full(n):
    
    return Matrix([Symbol("x_{" + ",".join(map(str, indices)) + "}") for indices in it.product(G, repeat=n)])

# but taking into account the fact that not all of them are accessible we proceed to the reduction
# according to which cycles orientations are not distinguishble (in the article we use C for the map from R^6 to R^5, 
#here we call it Cmap)
e6=eye(6)

Cmap = Matrix([e6[0,:],        #[1,0,0,0,0,0]
               e6[1,:],        #[0,1,0,0,0,0]
               e6[2,:],        #[0,0,1,0,0,0]
               e6[3,:],        #[0,0,0,1,0,0]
               e6[4,:]+e6[5,:] #[0,0,0,0,1,1]
          ])

def Cn(n):
#   C ⊗ C ⊗ ... ⊗ C (n times), a 5**n x 6**n matrix.
    return TP(*[Cmap] * n)


#the available invariants are listed in the following vector
def z(n):
    z = Cn(n)*x_full(n)
    return z

#Analogously we symbolically construct the vectors of all admissible values of correlation function
def y_patt(n):
    return Matrix([Symbol("y_{" + ",".join(map(str, indices)) + "}") for indices in it.product(Pi, repeat=n)])

#Then we build the coefficient matrix of the system y_patt= M(n)*z constructed party by party as follows
def SW_i(i):
    return SW.subs({Symbol(name): Symbol(f'{name}_{i}') for name in 'etcs'})

def M(n):
    """SW_i(1) ⊗ SW_i(2) ⊗ ... ⊗ SW_i(n)"""
    return TP(*[SW_i(i) for i in range(1, n + 1)])

In [14]:
# We show now how to construct the orbit space analysis and simplify the calculations exposed above

#First, we need to construct the matrix R_n^{sec} which is constructed starting from the projectors 
#PU and PW referred to monopartite analysis made above

def R_sec(n):
    cols = []
    # we construct the matrix whose columns are the sec vectors arranged in the block decompositions
    # by considering an index that runs over all possible number of parties including also 0
    for k in range(n + 1):
        #the next variable runs over all possible lists generated by picking up n objects and rearranging them
        #in subsets of size k where order does not matter, or in other words Wpos is the index that takes 
        #into account all possible postions of party in subspace W
        for Wpos in it.combinations(range(n), k):
            P = sum(
                (
                    #here when i in Wpos that means party i lies in W, then one tensor factor is PW*gam[g]*PW (PU otherwise)
                    TP(*[  PW*gam[g]*PW if i in Wpos else PU  for i in range(n)  ])
                    for g in G
                ),
                zeros(5**n, 5**n)
            ) / 6
            for v in P.columnspace():
                norm=0
                for i in range(len(v)):
                    if v[i] != 0:
                        norm += 1
                cols.append(norm * v)

    return Matrix.hstack(*cols)

#One can take the invariant projector and 
#gain easily its linearly independent columns that are NOT rearranged in such a way that they are organized with 
#respect to the decompostion R[\Pi]^{\otimes n}=(U \oplus W)^{\otimes n} but they are organized w.r. to orbit construction

def P(n): # projector onto the invariant subspace of n-parties equality pattern space
    P = sum((TP(*[gam[g]] * n) for g in G), zeros(5**n, 5**n)) / 6
    return P

def R_orb(n):
    Rcols = []
    for v in P(n).columnspace():           # one column per orbit (pivot columns), ordered by the
                                           # lexicographically first index tuple of each orbit,
                                           # w.r.t. the basis order (D, T1, T2, T3, Eq) used in gam
        for i in range(len(v)):
            if v[i] != 0:
                norm_1 = v[i]**(-1)
                break
        Rcols.append(norm_1 * v)
    return Matrix.hstack(*Rcols)

# Vectors appearing as columns of R_orb are not normalized but are orthogonal and this is the reason why in the formula of L_orb we can 
# assume that (R_orb.T * R_orb) is diagonal

def L_orb(n):
    
    return Matrix.hstack(*P(n).columnspace()).T

# Here we perform the reduction by multiplying by matrices L and R, obtatining the square matrix acting inside the orbit space (which is not the tensor product anymore)
def M_orb(n):
    return L_orb(n) * M(n) * R_orb(n) # y_patt = M_orb z_orb

# then whe consider the transition matrix to go from sec analysis to orbit analysis
def Q(n):
    return L_orb(n) * R_sec(n)

# Finally we see how coefficient matrix in orbit space is with respect to the above transformation (denoted as Q in the article)
def M_sec(n):
    Qn=Q(n)
    Matrix_sec = Qn.inv() * M_orb(n) * Qn # y_sec = M z
    
    subs_dict = {Symbol(f'e_{i}') + 3*Symbol(f't_{i}') + 2*Symbol(f'c_{i}'): Symbol(f's_{i}')
        for i in range(1, n + 1)}
    
    return Matrix_sec.applyfunc(factor).subs(subs_dict)